In [6]:
import os
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

In [7]:
# Overall frequency span represented by the receiver.
FREQ_MIN_MHZ = 0
FREQ_MAX_MHZ = 18_000

# The supplied SCAN receiver has 500 MHz instantaneous
# bandwidth and dwell centres:
# 250, 750, 1250, ..., 17750 MHz
# Therefore we represent the environment using 36 possible
# 500-MHz receiver observation windows:
# [0,500), [500,1000), ..., [17500,18000)
FREQ_BIN_MHZ = 500

# The supplied SCAN configuration contains 50 ms and
# 100 ms dwell durations.
# We use the finest observed dwell duration as our initial
# time resolution.
# This is an engineering/design choice, not a PS requirement.
TIME_BIN_S = 0.050

FREQ_EDGES_MHZ = np.arange(
    FREQ_MIN_MHZ,
    FREQ_MAX_MHZ + FREQ_BIN_MHZ,
    FREQ_BIN_MHZ,
    dtype=np.float32
)

FREQ_CENTRES_MHZ = (
    FREQ_EDGES_MHZ[:-1] + FREQ_BIN_MHZ / 2
)

N_FREQ_BINS = len(FREQ_CENTRES_MHZ)

print(f"Frequency range : {FREQ_MIN_MHZ}–{FREQ_MAX_MHZ} MHz")
print(f"Frequency bin   : {FREQ_BIN_MHZ} MHz")
print(f"Number of bands : {N_FREQ_BINS}")
print(f"Time resolution : {TIME_BIN_S * 1000:.0f} ms")
print()
print("Centres:", FREQ_CENTRES_MHZ)

Frequency range : 0–18000 MHz
Frequency bin   : 500 MHz
Number of bands : 36
Time resolution : 50 ms

Centres: [  250.   750.  1250.  1750.  2250.  2750.  3250.  3750.  4250.  4750.
  5250.  5750.  6250.  6750.  7250.  7750.  8250.  8750.  9250.  9750.
 10250. 10750. 11250. 11750. 12250. 12750. 13250. 13750. 14250. 14750.
 15250. 15750. 16250. 16750. 17250. 17750.]


In [8]:
# Folder containing STARE training HDF5 files
TRAIN_DATA_DIR = Path("./train_data")

# Folder where RF environments will be saved
TRAIN_RFENV_DIR = Path("./train_RFEnvs")

TRAIN_RFENV_DIR.mkdir(parents=True, exist_ok=True)

print("Input :", TRAIN_DATA_DIR.resolve())
print("Output:", TRAIN_RFENV_DIR.resolve())

Input : C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\train_data
Output: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\train_RFEnvs


In [9]:
h5_files = sorted(TRAIN_DATA_DIR.glob("*.h5"))
print(f"Found {len(h5_files)} HDF5 files.")

Found 5 HDF5 files.


In [10]:
def create_rf_environment(h5_path):
    """
    Convert one STARE HDF5 scenario into a binary
    time × frequency RF environment.

    Output:
        rf_env[t, f] = 1
            if at least one pulse occurs in frequency band f
            during time slot t

        rf_env[t, f] = 0
            otherwise
    """

    with h5py.File(h5_path, "r") as f:

        data = f["data"][:]

        # PDW columns
        toa_us = data[:, 0]
        frequency_mhz = data[:, 1]

    # --------------------------------------------------------
    # Convert time from microseconds → seconds
    # --------------------------------------------------------

    toa_s = toa_us / 1_000_000.0

    # --------------------------------------------------------
    # Determine number of time bins
    # --------------------------------------------------------

    max_time_s = np.max(toa_s)

    n_time_bins = int(
        np.ceil(max_time_s / TIME_BIN_S)
    )

    # --------------------------------------------------------
    # Convert every pulse into:
    #
    #     time_bin
    #     frequency_bin
    # --------------------------------------------------------

    time_indices = np.floor(
        toa_s / TIME_BIN_S
    ).astype(np.int64)

    freq_indices = np.floor(
        frequency_mhz / FREQ_BIN_MHZ
    ).astype(np.int64)

    # --------------------------------------------------------
    # Keep only frequencies represented by our 36 bands.
    # --------------------------------------------------------

    valid = (
        (time_indices >= 0)
        & (time_indices < n_time_bins)
        & (freq_indices >= 0)
        & (freq_indices < N_FREQ_BINS)
    )

    time_indices = time_indices[valid]
    freq_indices = freq_indices[valid]

    # --------------------------------------------------------
    # Create binary RF environment
    # --------------------------------------------------------

    rf_env = np.zeros(
        (n_time_bins, N_FREQ_BINS),
        dtype=np.uint8
    )

    # Every observed pulse activates its corresponding
    # time-frequency cell.
    rf_env[time_indices, freq_indices] = 1

    return rf_env

In [11]:
for i, h5_path in enumerate(h5_files, start=1):

    output_path = TRAIN_RFENV_DIR / f"{h5_path.stem}.npy"

    if output_path.exists():
        print(f"[{i}/{len(h5_files)}] SKIP: {h5_path.name}")
        continue

    rf_env = create_rf_environment(h5_path)

    np.save(output_path, rf_env)

    print(
        f"[{i}/{len(h5_files)}] "
        f"{h5_path.name} → {rf_env.shape}"
    )

print("\nDone.")

[1/5] SKIP: config_0.h5
[2/5] SKIP: config_1.h5
[3/5] SKIP: config_10.h5
[4/5] SKIP: config_100.h5
[5/5] SKIP: config_1000.h5

Done.


In [12]:
#Test
"""rf_files = sorted(
    p for p in TRAIN_RFENV_DIR.glob("*.npy")
    if p.name != "environment_config.npy"
)

print(f"RF environments: {len(rf_files)}")

for path in rf_files[:10]:
    env = np.load(path)

    print(
        f"{path.name:30s} "
        f"shape={env.shape} "
        f"occupancy={env.mean():.2%}"
    )"""

'rf_files = sorted(\n    p for p in TRAIN_RFENV_DIR.glob("*.npy")\n    if p.name != "environment_config.npy"\n)\n\nprint(f"RF environments: {len(rf_files)}")\n\nfor path in rf_files[:10]:\n    env = np.load(path)\n\n    print(\n        f"{path.name:30s} "\n        f"shape={env.shape} "\n        f"occupancy={env.mean():.2%}"\n    )'

In [13]:
# Folder containing STARE validation HDF5 files
VAL_DATA_DIR = Path("./val_data")

# Folder where RF environments will be saved
VAL_RFENV_DIR = Path("./val_RFEnvs")

VAL_RFENV_DIR.mkdir(parents=True, exist_ok=True)

print("Input :", VAL_DATA_DIR.resolve())
print("Output:", VAL_RFENV_DIR.resolve())

Input : C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\val_data
Output: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\val_RFEnvs


In [14]:
h5_files = sorted(VAL_DATA_DIR.glob("*.h5"))
print(f"Found {len(h5_files)} HDF5 files.")

Found 4 HDF5 files.


In [15]:
for i, h5_path in enumerate(h5_files, start=1):

    output_path = VAL_RFENV_DIR / f"{h5_path.stem}.npy"

    if output_path.exists():
        print(f"[{i}/{len(h5_files)}] SKIP: {h5_path.name}")
        continue

    rf_env = create_rf_environment(h5_path)

    np.save(output_path, rf_env)

    print(
        f"[{i}/{len(h5_files)}] "
        f"{h5_path.name} → {rf_env.shape}"
    )

print("\nDone.")

[1/4] config_0.h5 → (535, 36)
[2/4] config_1.h5 → (850, 36)
[3/4] config_10.h5 → (598, 36)
[4/4] config_100.h5 → (593, 36)

Done.
